# Project 2.2: training runner

Trains the fusion models with the shared code on Colab (T4 GPU). It reads the feature hand-off written by `notebooks/02_data_and_baselines.ipynb` (`MyDrive/vizwiz_fusion/features`) and saves every run to `MyDrive/vizwiz_fusion/runs/<model>/seed<k>/`. Settings live in `configs/default.yaml`; formats and model names are in `docs/data_contract.md`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
REPO = "/content/vizwiz-answerability-fusion"
if not os.path.exists(REPO):
    !git clone -q https://github.com/vishnuvardhan0ffl/vizwiz-answerability-fusion.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1

## Quick check that the code works

In [ ]:
!python -m tests.test_smoke

## Train

Trains every model in `src/train.py` (`TRAINABLE`) plus late fusion, 3 seeds each. Runs that already have predictions (for example the baselines from the data notebook) are skipped. Features are copied to the Colab disk first, because reading them from Drive is slow.

To train one model only, replace `all` with its name, e.g. `--model gated_clip`.

In [ ]:
!python -m src.train --config configs/default.yaml --model all --skip-existing --cache-dir /content/features_local

## Results so far (dev AUPRC used for model selection)

In [ ]:
import glob, json
import pandas as pd
rows = [json.load(open(p)) for p in glob.glob("/content/drive/MyDrive/vizwiz_fusion/runs/*/seed*/summary.json")]
cols = ["model", "seed", "best_dev_auprc", "epochs_run", "sec_per_epoch", "params", "peak_vram_gb"]
pd.DataFrame(rows).reindex(columns=cols).sort_values(["model", "seed"])